#  Strategic Analysis of Superstore Performance
### Business Intelligence Report — US Superstore Dataset

> **Objective:** Translate business objectives into actionable insights through  
> diagnostic visualisations, geographic analysis, discount diagnostics, and  
> interactive exploration.

---
**Libraries used:** `pandas` · `numpy` · `matplotlib` · `seaborn` · `ipywidgets`  
**Dataset:** US Superstore (~9 994 transactions, 21 columns)


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, Dropdown, IntSlider
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

# ── Global style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#f9f9f9',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'DejaVu Sans',
})
PALETTE = sns.color_palette("Set2")
print("✅ Imports OK")


## 2. Data Scoping & Preparation

### 2.1 Load & Assess


In [ ]:
df = pd.read_csv('Sample_-_Superstore.csv', encoding='latin1')

print(f"Shape: {df.shape}")
print(f"\nColumns:\n{df.columns.tolist()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
df.describe()


### 2.2 Cleaning

**Decisions:**
- **Duplicates:** dropped with `drop_duplicates()` — identical rows carry no extra information.
- **Postal Code:** zero missing values in this dataset; if any appeared we would `fillna(0)` since the column is numeric and used for geographic grouping only.
- **Date columns:** parsed to `datetime64` so we can extract year, month, and period objects for time-series grouping.


In [ ]:
# Remove duplicates
df = df.drop_duplicates()

# Parse dates
for col in ['Order Date', 'Ship Date']:
    df[col] = pd.to_datetime(df[col], dayfirst=False)

print("Data types after conversion:")
print(df[['Order Date', 'Ship Date']].dtypes)


### 2.3 Feature Engineering

In [ ]:
df['Profit Margin']    = (df['Profit'] / df['Sales']) * 100
df['Order Year']       = df['Order Date'].dt.year
df['Order Month']      = df['Order Date'].dt.month
df['Order Month-Year'] = df['Order Date'].dt.to_period('M')

print("Sample of engineered features:")
df[['Sales', 'Profit', 'Profit Margin', 'Order Year', 'Order Month']].head()


## 3. Deep-Dive Exploratory Analysis (Matplotlib)

### 3.1 Time-Series Trend Analysis

An interactive dropdown lets you filter by product category.  
Look for **seasonality** (Q4 spikes) and **year-over-year growth**.


In [ ]:
monthly_sales = df.groupby(['Order Month-Year', 'Category'])['Sales'].sum().reset_index()
monthly_sales['Date'] = monthly_sales['Order Month-Year'].dt.to_timestamp()

def plot_monthly_sales(category='All'):
    fig, ax = plt.subplots(figsize=(13, 5))
    if category == 'All':
        data = df.groupby('Order Month-Year')['Sales'].sum()
        ax.plot(data.index.to_timestamp(), data.values,
                marker='o', linewidth=2, markersize=4, color=PALETTE[0])
        ax.set_title('Monthly Sales Trend — All Categories', fontsize=15, fontweight='bold')
    else:
        data = monthly_sales[monthly_sales['Category'] == category]
        ax.plot(data['Date'], data['Sales'],
                marker='o', linewidth=2, markersize=4, color=PALETTE[1])
        ax.set_title(f'Monthly Sales Trend — {category}', fontsize=15, fontweight='bold')
    ax.set_xlabel('Date'); ax.set_ylabel('Sales ($)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()

categories = ['All'] + sorted(df['Category'].unique().tolist())
interact(plot_monthly_sales, category=Dropdown(options=categories, value='All', description='Category:'));


### 3.2 Geographic Sales Performance

A Top-N slider reveals whether sales are **centralised** around a few key states  
or **distributed** evenly across the country.


In [ ]:
state_sales = df.groupby('State')['Sales'].sum().sort_values(ascending=True)

def plot_top_states(top_n=10):
    fig, ax = plt.subplots(figsize=(12, max(6, top_n * 0.45)))
    top = state_sales.tail(top_n)
    colors = sns.color_palette("Blues_r", len(top))
    bars = ax.barh(range(len(top)), top.values, color=colors)
    ax.set_yticks(range(len(top))); ax.set_yticklabels(top.index)
    ax.set_xlabel('Total Sales ($)'); ax.set_title(f'Top {top_n} States by Sales', fontsize=14, fontweight='bold')
    for i, v in enumerate(top.values):
        ax.text(v + top.max() * 0.01, i, f'${v:,.0f}', va='center', fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
    plt.tight_layout(); plt.show()
    print(f"Top {top_n} states → ${top.sum():,.0f} | {top.sum()/state_sales.sum()*100:.1f}% of total sales")

interact(plot_top_states, top_n=IntSlider(min=5, max=25, value=10, description='Top N:'));


## 4. Communicating Insights (Seaborn)

### 4.1 Top 10 Most Profitable Products


In [ ]:
product_profit = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(x=product_profit.values, y=product_profit.index,
            palette='viridis', ax=ax, orient='h')
ax.set_title('Top 10 Most Profitable Products\nExecutive Summary — Product Performance',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Total Profit ($)'); ax.set_ylabel('')
for i, v in enumerate(product_profit.values):
    ax.text(v + product_profit.max() * 0.01, i, f'${v:,.0f}', va='center', fontsize=9, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout(); plt.show()

print(f"• Top product profit  : ${product_profit.iloc[0]:,.0f}")
print(f"• Top-10 combined     : ${product_profit.sum():,.0f}")
print(f"• Average per product : ${product_profit.mean():,.0f}")


### 4.2 Discount vs Profit — Scatter Analysis

A regression line (red dashed) shows the overall trend.  
The **break-even line** (y = 0) is the critical threshold.


In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category',
                alpha=0.55, s=40, ax=ax, palette='Set2')
sns.regplot(data=df, x='Discount', y='Profit', scatter=False,
            color='crimson', line_kws={'linewidth': 2, 'linestyle': '--'}, ax=ax)
ax.axhline(0, color='black', linewidth=1, alpha=0.4)
ax.text(0.52, 30, 'Break-even line', fontsize=10, alpha=0.6)
ax.set_title('Discount Strategy Analysis: Impact on Profitability by Category',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Discount Rate'); ax.set_ylabel('Profit ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()

high_disc = df[df['Discount'] > 0.2]
print(f"Transactions with >20% discount : {len(high_disc):,}")
print(f"Avg profit (>20% discount)      : ${high_disc['Profit'].mean():.2f}")
print(f"Loss rate  (>20% discount)      : {(high_disc['Profit'] < 0).mean()*100:.1f}%\n")
for cat in df['Category'].unique():
    hd = df[(df['Category'] == cat) & (df['Discount'] > 0.2)]
    if len(hd):
        print(f"  {cat}: avg profit @ >20% disc = ${hd['Profit'].mean():.2f}")


## 5. Methodology — Matplotlib vs Seaborn

| Criterion | Matplotlib | Seaborn |
|---|---|---|
| **Control** | Full, pixel-level | High-level defaults |
| **Speed (simple plot)** | Faster | Slight overhead |
| **Statistical built-ins** | Manual | Built-in (regplot, boxplot…) |
| **ipywidgets integration** | Native | Via matplotlib backend |
| **Publication aesthetics** | Requires manual styling | Ready out-of-the-box |
| **Best for** | Interactive dashboards, custom layouts | Stakeholder charts, statistical charts |

**Recommendation:**  
- *Rapid exploration / dashboards* → **Matplotlib** (control + widget integration)  
- *Stakeholder presentations* → **Seaborn** (clean defaults + statistical overlays)


## 6. 📊 Multi-Chart Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('Superstore — Executive Dashboard', fontsize=17, fontweight='bold', y=1.01)

# ── Chart 1: Monthly sales trend ─────────────────────────────────────────────
ax1 = axes[0, 0]
monthly_total = df.groupby('Order Month-Year')['Sales'].sum()
ax1.plot(monthly_total.index.to_timestamp(), monthly_total.values,
         marker='o', linewidth=1.8, markersize=3, color=PALETTE[0])
ax1.set_title('Monthly Sales Trend'); ax1.set_xlabel(''); ax1.set_ylabel('Sales ($)')
ax1.tick_params(axis='x', rotation=40)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

# ── Chart 2: Sales & Profit by Category ──────────────────────────────────────
ax2 = axes[0, 1]
cat_data = df.groupby('Category')[['Sales', 'Profit']].sum()
x = range(len(cat_data))
w = 0.35
ax2.bar([i - w/2 for i in x], cat_data['Sales'], w, label='Sales', color=PALETTE[0])
ax2.bar([i + w/2 for i in x], cat_data['Profit'], w, label='Profit', color=PALETTE[1])
ax2.set_xticks(list(x)); ax2.set_xticklabels(cat_data.index)
ax2.set_title('Sales & Profit by Category'); ax2.legend()
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

# ── Chart 3: Top 10 States ────────────────────────────────────────────────────
ax3 = axes[1, 0]
top10 = state_sales.tail(10)
ax3.barh(range(len(top10)), top10.values,
         color=sns.color_palette("Blues_r", len(top10)))
ax3.set_yticks(range(len(top10))); ax3.set_yticklabels(top10.index, fontsize=9)
ax3.set_title('Top 10 States by Sales')
ax3.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

# ── Chart 4: Discount vs Profit by Category ───────────────────────────────────
ax4 = axes[1, 1]
for i, cat in enumerate(df['Category'].unique()):
    sub = df[df['Category'] == cat]
    ax4.scatter(sub['Discount'], sub['Profit'], label=cat, alpha=0.5, s=20, color=PALETTE[i])
ax4.axhline(0, color='black', linestyle='--', alpha=0.4)
ax4.set_xlabel('Discount'); ax4.set_ylabel('Profit ($)')
ax4.set_title('Discount vs Profit by Category'); ax4.legend(fontsize=9)
ax4.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.tight_layout(); plt.show()


## 7. Outlier Identification — Discount vs Profit

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
sns.scatterplot(data=df, x='Discount', y='Profit', hue='Category',
                alpha=0.5, s=40, ax=ax, palette='Set2')
ax.axhline(0, color='black', linewidth=1, alpha=0.3)

top3    = df.nlargest(3, 'Profit')
bottom3 = df.nsmallest(3, 'Profit')

for _, row in top3.iterrows():
    ax.annotate(f"Best\n${row['Profit']:,.0f}",
                xy=(row['Discount'], row['Profit']),
                xytext=(15, -20), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#d4edda', edgecolor='green', alpha=0.85),
                arrowprops=dict(arrowstyle='->', color='green'))

for _, row in bottom3.iterrows():
    ax.annotate(f"Worst\n${row['Profit']:,.0f}",
                xy=(row['Discount'], row['Profit']),
                xytext=(15, 20), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#f8d7da', edgecolor='red', alpha=0.85),
                arrowprops=dict(arrowstyle='->', color='red'))

ax.set_title('Discount vs Profit — Outlier Identification', fontsize=14, fontweight='bold')
ax.set_xlabel('Discount'); ax.set_ylabel('Profit ($)')
ax.legend(title='Category', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout(); plt.show()


## 8. Bonus — Sub-Category Profit Margin Heatmap

In [ ]:
pivot = df.groupby(['Category', 'Sub-Category'])['Profit Margin'].mean().unstack(level=0)

fig, ax = plt.subplots(figsize=(8, 9))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Avg Profit Margin (%)'})
ax.set_title('Average Profit Margin by Sub-Category & Category\n(red = loss, green = gain)',
             fontsize=13, fontweight='bold')
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout(); plt.show()


## 9. 📝 Executive Summary

In [ ]:
total_sales  = df['Sales'].sum()
total_profit = df['Profit'].sum()
margin       = total_profit / total_sales * 100
top_state    = state_sales.index[-1]
top_state_s  = state_sales.iloc[-1]
top_cat      = df.groupby('Category')['Sales'].sum().idxmax()
top_product  = df.groupby('Product Name')['Profit'].sum().idxmax()
hd_loss_rate = (df[df['Discount'] > 0.2]['Profit'] < 0).mean() * 100

print("=" * 60)
print("  EXECUTIVE SUMMARY — US SUPERSTORE")
print("=" * 60)
print(f"\n📊 BUSINESS PERFORMANCE")
print(f"  • Total Revenue        : ${total_sales:>12,.0f}")
print(f"  • Total Profit         : ${total_profit:>12,.0f}")
print(f"  • Overall Profit Margin: {margin:.1f}%")

print(f"\n🗺️  GEOGRAPHIC PERFORMANCE")
print(f"  • Top state            : {top_state} (${top_state_s:,.0f})")
top5_share = state_sales.tail(5).sum() / total_sales * 100
print(f"  • Top 5 states share   : {top5_share:.1f}% of total sales")

print(f"\n🏆 PRODUCT PERFORMANCE")
print(f"  • Leading category     : {top_cat}")
print(f"  • Most profitable item : {top_product}")

print(f"\n💰 DISCOUNT STRATEGY")
print(f"  • >20% discount loss rate: {hd_loss_rate:.1f}% of transactions result in losses")
print(f"  • Recommended cap        : ≤20% to preserve margin")

print("\n" + "=" * 60)
print("KEY RECOMMENDATIONS")
print("=" * 60)
print("  1. Cap discounts at 20% — especially in Furniture & Office Supplies")
print("  2. Invest marketing budget in California, New York, Texas")
print("  3. Expand high-margin Technology sub-categories (Copiers, Phones)")
print("  4. Review Tables sub-category — consistently negative margins")
print("  5. Introduce discount-approval workflow for orders above 30% off")
